# Medias Móviles en 2020
# Este notebook ejecuta el método de medias móviles sobre los datos del IBEX35
# Objetivo: calcular SMA/EMA y señales durante el año 2020 para su análisis y comparación.

In [1]:
# Instalar dependencias si se ejecuta en Google Colab
import sys
if 'google.colab' in sys.modules:
    !pip install yfinance pandas_ta matplotlib

# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from load_data import DataProcessor
from medias_moviles import generate_ma_signals

plt.style.use('seaborn')
print('Librerías importadas correctamente')

ModuleNotFoundError: No module named 'load_data'

In [2]:
# Cargar datos usando DataProcessor y filtrar 2020
processor = DataProcessor()
df = processor.download_data()
processor.add_features()

# Filtrar solo 2020
df_2020 = df.loc['2020-01-01':'2020-12-31'].copy()
print('Filas en 2020:', df_2020.shape[0])
df_2020.head()

NameError: name 'DataProcessor' is not defined

In [ ]:
# Definir y ejecutar medias móviles sobre 2020
# Usamos SMA 50/200 por defecto pero se pueden ajustar
short_window = 50
long_window = 200

# Generar columnas de medias y señales
df_ma = generate_ma_signals(df_2020, short_window=short_window, long_window=long_window, column='Close', ma_type='SMA')

df_ma[['Close','MA_short','MA_long','signal']].tail()

In [ ]:
# Visualizar Close y Medias Móviles, marcando cruces
import matplotlib.dates as mdates

fig, ax = plt.subplots(figsize=(14,6))
ax.plot(df_ma.index, df_ma['Close'], label='Close')
ax.plot(df_ma.index, df_ma['MA_short'], label=f'MA_{short_window}')
ax.plot(df_ma.index, df_ma['MA_long'], label=f'MA_{long_window}')

# Detectar cruces (diff==2 o -2)
bull = df_ma[df_ma['signal_change'] == 2]
bear = df_ma[df_ma['signal_change'] == -2]
ax.scatter(bull.index, df_ma.loc[bull.index,'Close'], marker='^', color='green', label='Cruce alcista')
ax.scatter(bear.index, df_ma.loc[bear.index,'Close'], marker='v', color='red', label='Cruce bajista')

ax.set_title(f'IBEX - 2020: Close y MA {short_window}/{long_window}')
ax.legend()
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
plt.show()

In [ ]:
# Analizar rendimiento simple: estrategia basada en la señal vs buy-and-hold
# Retornos diarios

df_ma['ret'] = df_ma['Close'].pct_change().fillna(0)
# Estrategia: posición = signal del día anterior
df_ma['strategy_ret'] = df_ma['ret'] * df_ma['signal'].shift(1).fillna(0)

df_ma['cum_strategy'] = (1 + df_ma['strategy_ret']).cumprod()
df_ma['cum_buyhold'] = (1 + df_ma['ret']).cumprod()

print('Número aproximado de señales (cruces):', int(df_ma['signal_change'].abs().sum()/2))

fig, ax = plt.subplots(figsize=(12,5))
ax.plot(df_ma.index, df_ma['cum_buyhold'], label='Buy & Hold')
ax.plot(df_ma.index, df_ma['cum_strategy'], label='MA Strategy')
ax.legend()
ax.set_title('Rendimiento acumulado 2020')
plt.show()

In [ ]:
# Exportar resultados a CSV
output_path = 'notebooks/medias_moviles_2020_results.csv'
df_ma.to_csv(output_path)
print(f'Resultados guardados en {output_path}')

# Conclusión breve
# Este notebook muestra cómo calcular medias móviles y señales en 2020.
# Próximos pasos sugeridos:
# - Probar diferentes pares de ventanas (short/long)
# - Evaluar métricas más robustas (Sharpe, max drawdown)
# - Comparar con las anomalías detectadas por el CNN-AE
